# 01 · Load the match data

Reads every provided asset into numpy/pandas, aligned so that **row `i` = video frame `i` = row `i` of a submission**.
The loading code lives in `possession/load_data.py`; this notebook is where we look at it.

- Coordinates are in **metres**, origin at a pitch corner: x along the length (0–105.3), y across (0–68).
- `NaN` = not tracked (ball or player).
- First run parses the tracking (~3 s) and caches it in `cache/` (gitignored, never commit it).

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
# data folder: $MTAH_DATA_DIR if set, else ./data inside the repo, else ../data next to it
if "MTAH_DATA_DIR" not in os.environ:
    os.environ["MTAH_DATA_DIR"] = str(ROOT / "data" if (ROOT / "data" / "tactical").exists() else ROOT.parent / "data")
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "starter"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from possession.load_data import load_match, summary, HOME, AWAY
from utils import draw_pitch

%load_ext autoreload
%autoreload 2
print("data dir:", os.environ["MTAH_DATA_DIR"])

## Load

In [ ]:
m = load_match()
summary(m)

## What's in `m`

| attribute | shape | meaning |
|---|---|---|
| `ball_xy` | (N, 2) | ball position, metres |
| `ball_speed` | (N,) | as delivered, mostly NaN → recompute from positions |
| `players_xy` | (N, 46, 2) | every track slot (a tracklet, **not** a person) |
| `players_speed` | (N, 46) | m/s |
| `track_team` | (46,) | 0 = home Barcelona, 1 = away Athletic |
| `half`, `graded` | (N,) | 1/2 on scored frames, 0 otherwise |
| `tags` | DataFrame | Smart Tagging, `start_frame`/`end_frame` half-open, `team_side` = H/A/NaN (`side` is the Left/Central/Right tag) |
| `homography`, `homography_valid` | (N,3,3), (N,) | pitch (0–1) → pixels |
| `gt_public` | (N,) | H/A/D on frames 11185–18684, X elsewhere — **checking only, never tuning** |

In [ ]:
{k: getattr(m, k).shape for k in ["ball_xy", "ball_speed", "players_xy", "players_speed",
                                   "track_team", "half", "graded", "homography", "gt_public"]}

## A per-frame table (handy for quick looks)

In [ ]:
frames = pd.DataFrame({
    "half": m.half,
    "ball_x": m.ball_xy[:, 0], "ball_y": m.ball_xy[:, 1],
    "ball_seen": ~np.isnan(m.ball_xy[:, 0]),
    "n_home": (~np.isnan(m.players_xy[:, m.track_team == HOME, 0])).sum(1),
    "n_away": (~np.isnan(m.players_xy[:, m.track_team == AWAY, 0])).sum(1),
    "gt": m.gt_public,
})
frames.loc[11185:11195]

## One frame on the pitch

In [ ]:
def plot_frame(i, ax=None):
    ax = draw_pitch(ax, *m.pitch)
    xy = m.players_xy[i]
    for team, colour in [(HOME, "#d62728"), (AWAY, "#1f77b4")]:
        sel = m.track_team == team
        ax.scatter(xy[sel, 0], xy[sel, 1], s=80, c=colour, edgecolor="k", zorder=3)
    if not np.isnan(m.ball_xy[i, 0]):
        ax.scatter(*m.ball_xy[i], s=40, c="white", edgecolor="k", zorder=4)
    ax.invert_yaxis()   # Metrica y = 0 is the top of the image: match the camera view
    ax.set_title(f"frame {i}  ·  GT {m.gt_public[i]}  ·  ball {'seen' if not np.isnan(m.ball_xy[i,0]) else 'MISSING'}")
    return ax

plot_frame(11185 + 25 * 30);  

## Where the ball goes missing
The single most important fact about the data. Per-minute share of frames with a ball position, over the graded play.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharey=True)
for ax, h in zip(axes, (1, 2)):
    sel = frames[frames.half == h]
    per_min = sel.ball_seen.groupby((sel.index - sel.index[0]) // (25 * 60)).mean()
    ax.bar(per_min.index, per_min.values, color="#555")
    ax.set_title(f"half {h}: share of frames with the ball tracked, per minute")
    ax.set_ylim(0, 1)
axes[-1].set_xlabel("minute of the half")
plt.tight_layout()

In [ ]:
# Length of each continuous ball gap on graded frames
seen = frames.ball_seen.values & m.graded
edges = np.diff(np.r_[0, (~seen & m.graded).astype(int), 0])
gaps = (np.flatnonzero(edges == -1) - np.flatnonzero(edges == 1)) / 25
print(f"{len(gaps)} gaps · median {np.median(gaps):.2f}s · 90th pct {np.percentile(gaps, 90):.1f}s · longest {gaps.max():.0f}s")
print(f"share of missing time in gaps > 1 s: {gaps[gaps > 1].sum() / gaps.sum():.0%}")
plt.hist(np.clip(gaps, 0, 20), bins=80, color="#555"); plt.yscale("log")
plt.xlabel("gap length (s, clipped at 20)"); plt.ylabel("count");

## The public 5 minutes next ball location x and y
Understanding what H / A / D look like in the data

In [ ]:
win = slice(11185, 18685)
t = (np.arange(win.start, win.stop) - win.start) / 25
colour = {"H": "#d62728", "A": "#1f77b4", "D": "#999999"}

fig, axes = plt.subplots(4, 1, figsize=(15, 8), sharex=True, gridspec_kw={"height_ratios": [1, 1, 2, 2]})
axes[0].scatter(t, np.zeros_like(t), c=[colour[l] for l in m.gt_public[win]], marker="|", s=400)
axes[0].set_title("ground truth (red H, blue A, grey D)"); axes[0].set_yticks([])
axes[1].scatter(t, np.zeros_like(t), c=np.where(frames.ball_seen[win], "k", "#eee"), marker="|", s=400)
axes[1].set_title("ball tracked (black) / missing"); axes[1].set_yticks([])
axes[2].plot(t, m.ball_xy[win, 0], ".", ms=1, c="k")
axes[2].set_ylabel("ball x (m)")
for x_line in (0, m.pitch[0]): axes[2].axhline(x_line, ls="--", lw=.8, c="grey")   # goal lines
axes[3].plot(t, m.ball_xy[win, 1], ".", ms=1, c="k")
for y_line in (0, m.pitch[1]): axes[3].axhline(y_line, ls="--", lw=.8, c="grey")   # touchlines
axes[3].set_ylabel("ball y (m)"); axes[3].set_xlabel("seconds since kick-off")
plt.tight_layout()

## Smart Tagging

In [ ]:
print(m.tags.code.value_counts().to_string())
m.tags.head(40)

## Smart Tagging on a timeline

Each row is one tag code. Bars are coloured by the tagged team (red Barcelona, blue Athletic, grey N/A).
For the defending codes the tagged team is the *defending* side; each one mirrors an attacking tag with the same interval and the other team.
Set pieces are fixed 35 s windows, so only their start (the restart) means anything.

In [ ]:
def plot_tags(start, stop, ax_tags, ax_gt=None):
    tags = m.tags[(m.tags.end_frame > start) & (m.tags.start_frame < stop)]
    codes = sorted(m.tags.code.unique())
    team_colour = {"H": "#d62728", "A": "#1f77b4"}
    for _, tag in tags.iterrows():
        row = codes.index(tag.code)
        a, b = max(tag.start_frame, start), min(tag.end_frame, stop)
        ax_tags.barh(row, (b - a) / 25, left=(a - start) / 25, height=0.7,
                     color=team_colour.get(tag.team_side, "#aaaaaa"), alpha=0.85)
    ax_tags.set_yticks(range(len(codes))); ax_tags.set_yticklabels([c.title() for c in codes], fontsize=8)
    ax_tags.invert_yaxis(); ax_tags.set_xlabel("seconds since frame {}".format(start))
    if ax_gt is not None:
        t = (np.arange(start, stop) - start) / 25
        labels = m.gt_public[start:stop]
        ax_gt.scatter(t, np.zeros_like(t), c=[colour.get(l, "white") for l in labels], marker="|", s=400)
        ax_gt.set_yticks([]); ax_gt.set_title("ground truth (red H, blue A, grey D)")

fig, (ax_gt, ax_tags) = plt.subplots(2, 1, figsize=(15, 7), sharex=True, gridspec_kw={"height_ratios": [1, 6]})
plot_tags(11185, 18685, ax_tags, ax_gt)
ax_tags.set_title("Smart Tagging, public 5 minutes")
plt.tight_layout()

## Smart Tagging over the whole match: how many tags, and for which team

In [ ]:
counts = (m.tags.assign(team=m.tags.team_side.fillna("N/A"))
          .groupby(["code", "team"]).size().unstack(fill_value=0)
          .reindex(columns=["H", "A", "N/A"], fill_value=0))
counts = counts.loc[counts.sum(axis=1).sort_values().index]
ax = counts.plot.barh(stacked=True, figsize=(10, 6), color=["#d62728", "#1f77b4", "#aaaaaa"])
ax.set_xlabel("number of tags"); ax.set_ylabel(""); ax.set_title("Smart Tagging codes by tagged team")
ax.legend(["Barcelona (H)", "Athletic (A)", "N/A"]);

### Smart Tagging on a timeline
One lane per tag code, coloured by the team the tag names (red Barcelona, blue Athletic, grey `N/A`). Defending tags name the *defending* team, so their colour is the team **without** the ball; they are exact mirrors of the attacking tags (same intervals). Set pieces are fixed 35 s windows: only their start (the restart) means anything.

In [ ]:
TEAM_COLOUR = {"H": "#d62728", "A": "#1f77b4"}
ORDER = ["BUILD UP", "PROGRESSION", "CREATING CHANCES", "ATTACKING TRANSITION", "LONG BALL",
         "BALL IN FINAL THIRD", "BALL IN THE BOX", "PLAYERS IN THE BOX",
         "DEFENDING IN ATTACKING THIRD", "DEFENDING IN MIDDLE THIRD", "DEFENDING IN DEFENSIVE THIRD",
         "DEFENSIVE TRANSITION", "SET PIECES", "GOALS"]

def plot_tags(ax, start, stop, minutes=True):
    sel = m.tags[(m.tags.end_frame > start) & (m.tags.start_frame < stop)]
    scale = 25 * 60 if minutes else 25
    for lane, code in enumerate(ORDER):
        rows = sel[sel.code == code]
        for r in rows.itertuples():
            colour = TEAM_COLOUR.get(r.team_side, "#999999")
            a, b = max(r.start_frame, start), min(r.end_frame, stop)
            ax.broken_barh([((a - start) / scale, (b - a) / scale)], (lane - 0.4, 0.8), color=colour, alpha=.8)
            if code == "SET PIECES":   # mark the restart moment
                ax.plot([(r.start_frame - start) / scale] * 2, [lane - .45, lane + .45], c="k", lw=1)
    ax.set_yticks(range(len(ORDER))); ax.set_yticklabels([c.title() for c in ORDER], fontsize=8)
    ax.invert_yaxis(); ax.set_xlim(0, (stop - start) / scale)

# whole match, one row per half
fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True)
for ax, (h, first, last) in zip(axes, [(1, 11185, 90651), (2, 110808, 190197)]):
    plot_tags(ax, first, last + 1)
    ax.set_title(f"Smart Tagging, half {h} (red Barcelona, blue Athletic, grey N/A; black tick = set-piece restart)")
axes[-1].set_xlabel("minutes since the start of the half")
plt.tight_layout()

### Zoom: the public 5 minutes, tags under the ground truth
For **understanding** only: how tag phases line up with real possessions (they are phases, not possessions, so expect them to be longer and to overlap turnovers).

In [ ]:
start, stop = 11185, 18685
fig, axes = plt.subplots(2, 1, figsize=(16, 7), sharex=True, gridspec_kw={"height_ratios": [1, 8]})
t = np.arange(stop - start) / 25
axes[0].scatter(t, np.zeros_like(t), c=[colour[l] for l in m.gt_public[start:stop]], marker="|", s=400)
axes[0].set_yticks([]); axes[0].set_title("ground truth (red H, blue A, grey D)")
plot_tags(axes[1], start, stop, minutes=False)
axes[1].set_xlabel("seconds since kick-off")
plt.tight_layout()